In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from cmdstanpy import CmdStanModel

import arviz as az

C:\Users\Josh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the 2026 dataset: https://memory.psych.upenn.edu/Data_Archive#2026

In [ ]:
# Get data
data = pd.read_csv('data/Exp3_AllData.csv', header=0, sep=',')
data.head()
data = data[~np.isnan(data["Output_Rank"])]

# Convert to ints
data["Listnum"] = data["Listnum"].astype(int)
data["Spatial_Input_Order"] = data["Spatial_Input_Order"].astype(int)
data["Temporal_Input_Position"] = data["Temporal_Input_Position"].astype(int)
data["Spatial_Recall_Order"] = data["Spatial_Recall_Order"].astype(int)
data["Length"] = data["Length"].astype(int)
data["Correct"] = data["Correct"].astype(int)
data["Distance_From_Correct"] = data["Distance_From_Correct"].astype(int)
data["Serial_Pos_Encoded"] = data["Serial_Pos_Encoded"].astype(int)
data["Output_Rank"] = data["Output_Rank"].astype(int)

# Filter for only spatial and varied data
data = data[data["Order_Type"] == 'Spatial']
data = data[data["List_Type"] == 'Varied']

# Only use a fraction of the data
train_percent = 0.05
train_n = int(data.shape[0]*train_percent)
data = data.sample(n=train_n)

data_dict = {
    # Sizes
    "N": data.shape[0],
    "S": data["Uniqueid"].nunique(),
    # IDs
    "subject_index": data["Uniqueid"].astype("category").cat.codes + 1,
    # Experiment
    #"list_number": data.Listnum.values,
    "spatial_input_order": data.Spatial_Input_Order.values,
    #"temporal_input_pos": data.Temporal_Input_Position.values,
    "list_length": data.Length.values,
    # Results
    "spatial_recall_order": data.Spatial_Recall_Order.values,
    "correct": data.Correct.values,
    "distance_from_correct": data.Distance_From_Correct.values,
    #"output_rank": data.Output_Rank.values,
    "rt": data.Initial_RT_Time.values,
}

In [8]:
# Compile model
model = CmdStanModel(stan_file="stan/model.stan")

In [ ]:
# Sample model
fit = model.sample(data=data_dict, chains=4, iter_sampling=2500, iter_warmup=1000)

# Display sampling diagnostics
print(fit.diagnose())

15:01:39 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]





chain 1:   3%|▎         | 100/3500 [01:37<55:08,  1.03it/s, (Warmup)]




chain 1:   6%|▌         | 200/3500 [02:24<37:19,  1.47it/s, (Warmup)]


chain 1:   9%|▊         | 300/3500 [03:04<29:18,  1.82it/s, (Warmup)]




chain 1:  11%|█▏        | 400/3500 [04:46<38:00,  1.36it/s, (Warmup)]


